<a href="https://colab.research.google.com/github/MamoMGD1/ML_101/blob/main/notebooks/Section06_Natural_Language_Processing/Chapter02_Large_Language_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **LANGUAGE MODELS**

In the previous chapter, we built a solid foundation on one of the most magical concepts in NLP: the **Transformer**. Just as autoencoders became a groundbreaking idea in Computer Vision which are powerful enough that each of their individual parts (encoder and decoder) could be repurposed for a wide variety of tasks. We could use the **encoder** part of a transformer for language models that we call **Representative LMs** (like BERT) and the **decoder** part for **Generative LMs** (like GPT).

With all their core components like **embeddings, positional encodings, attention mechanisms, and more...** transformers have the capacity to learn languages in a way that closely resembles how we humans process them. The models built upon transformers are what we nowadays call **Language Models**.  

A **language model** is any model that can assign probabilities to sequences of words within a given context. For example, given the sequence:  

> *"The capital of France is ______."*  

a well-trained language model should confidently predict the word *"Paris"* as the most likely continuation.  

Training such a model requires time, immense data, and a series of essential steps. But the outcome is astonishing: a system that can not only complete sentences but also reason, answer questions, and even generate entirely new text.  

In this chapter, we’ll shine a light on some of the most influential language model architectures that emerged after the landmark 2017 paper *“Attention Is All You Need”*. These models have reshaped modern AI and continue to define how we interact with technology today.

# **LARGE LANGUAGE MODELS**

The modern era of AI is deeply tied to a curious observation in machine learning known as the **double descent phenomenon**. Traditionally, we were taught that making models too complex leads to overfitting — the model memorizes instead of learning. Yet experiments showed something counterintuitive: once the model becomes *extremely* overparameterized (far more parameters than data points), the performance curve can dip back down, leading to **better generalization at larger scales**.  

This finding revealed something profound: in **high-dimensional spaces**, the rules of learning behave differently. When models grow large enough, they don’t just memorize — they discover smoother, more robust structures in data. What seemed like “too much capacity” in smaller models becomes a source of **stability and expressiveness** in larger ones.  

This realization sparked a shift in perspective. If bigger models generalize better, what would happen if we kept pushing the scale? Could the same network, trained on raw text at a massive scale, capture the complexities of language, reasoning, and knowledge — not by task-specific rules, but by sheer exposure to data and parameters?  

That question was the seed of **Large Language Models**. Researchers began to explore whether scaling up the size of language models (more layers, more hidden units, more training data) would not just improve accuracy, but fundamentally change *what the model could do*.  

<div align="center">
  <img src="https://raw.githubusercontent.com/MamoMGD1/ML_101/main/images/S06_C02/mooreslaw.png" width="600">
</div>

**Image Credit:** [Original Website](https://medium.com/@athudi29/the-ultimate-guide-to-llm-and-local-llm-ef80524b9a1e)

This idea opened the door to a new era of NLP, where instead of building specialized systems for each task, we build **general-purpose models** at unprecedented scales like GPT.  

## **BERT (Oct 2018, Google)**

The famous paper again from Google published in October 2018, [**BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding**](https://arxiv.org/abs/1810.04805), was one of the first major advances right after *“Attention Is All You Need”* in the ML/NLP world. The architecture—**Bidirectional Encoder Representations from Transformers** (BERT)—is essentially the Transformer encoder stack trained on massive text corpora.  

The original BERT was trained on:  
- **Wikipedia (English)** – about 2,5 billion words  
- **BookCorpus** – about 800 million words  

One of its key distinguishing features is that it uses an **encoder-only architecture**, and learns from **huge training data** rather than task-specific supervision.  

But what does the *“Bidirectional”* in its name actually mean?  

- In unidirectional (autoregressive) models, the model predicts each word based only on its **past context** (the words before it).  
- BERT did something different: it randomly masks some words in the sentence with a special token `[MASK]`, and the model must predict those words using **both** the words before *and* the words after the mask. Thus, it is **bidirectional** in its context window.

This is much like a “fill-in-the-blank” exercise children solve in school.  

Now, having introduced what BERT does and why it matters, let’s dive deeper into its architecture.

<div align="center">
  <img src="https://raw.githubusercontent.com/MamoMGD1/ML_101/main/images/S06_C02/bert.png" width="600">
  <h5>Figure 1: BERT As Transformers' Encoder</h5>
</div>

**Image Credit:** [Original Website](https://kambale.dev/fine-tuning-bert)


### **BERT-Base Architecture**

* **Encoder Blocks (Layers)**: BERT-Base is built from **12 stacked Transformer encoder blocks**.

* **Self-Attention Heads**: Inside each encoder block, there are **12 self-attention heads**.

* **Hidden Size (Embedding Dimension)**: Each token in the input sequence is represented as a **768-dimensional vector**.

* **Feed-Forward Network Size**: After self-attention, each encoder block contains a feed-forward neural network with an **inner dimension of 3072**. This expansion layer (roughly 4× the hidden size) allows the model to perform complex non-linear transformations before projecting back down to 768 dimensions.

* **Maximum Input Sequence Length**: BERT-Base can process sequences of up to **512 tokens**.

* **Total Parameters**: In total, BERT-Base contains about **110 million parameters**.

### **Pre-Training Phase**

As we discussed, BERT is built from **12 stacked Transformer encoder blocks**. Each encoder is designed to refine token embeddings into a richer representation of meaning. While the inputs to BERT are already embedded word vectors, the outputs of the encoders go much further. They integrate:

* **Word meaning** (semantic embedding of the token itself)
* **Position information** (via positional encoding)
* **Contextual effects** (captured through self-attention across all other words in the sequence)

So, the representation of the word *“mouse”* at the output of the encoder is not just “mouse” in isolation. Instead, it becomes *“mouse” as an electronic device, appearing at the end of the sentence (likely as the object), functioning as a noun, and modified by the adjective ‘wireless’.* This layered contextualization is what makes the encoder’s outputs so powerful. And when **12 of these encoders** are stacked, the model gains an extraordinary ability to understand nuanced language.

But here lies a challenge: BERT was trained on massive unlabelled datasets — namely the **entire English Wikipedia (2.5 billion words)** and the **BooksCorpus (800 million words)**. Without explicit labels, how can such a model be trained?

The solution was **Self-Supervised Learning** (more on it in `Section07`). This approach cleverly turns unlabelled data into supervised learning tasks without requiring human annotation. For BERT, two self-supervised objectives were central to pre-training: **Next Sentence Prediction (NSP)** and **Masked Language Modeling (MLM)**

---

### **NSP (Next Sentence Prediction)**

In NSP, the model is given *two* text segments (Sentence A and Sentence B) concatenated into a single input of the form:

\[CLS]  Sentence A  \[SEP]  Sentence B  \[SEP]

- The special token `[CLS]` is placed **at the beginning** of the whole input.  
- After processing the full input with the encoder stack, the model takes the **final hidden vector corresponding to the `[CLS]` token** and feeds it to a small classification head (a linear layer + softmax) that outputs a probability over two labels: **IsNext (1)** or **NotNext (0)**.  
- During pretraining, pairs are constructed so that **50%** of pairs are *positive* (Sentence B truly follows Sentence A in the corpus) and **50%** are *negative* (Sentence B is a random sentence sampled from elsewhere). This makes NSP a *self-supervised binary classification* problem: it needs no human labeling because we can create positives and negatives automatically.

#### **Why it was used**  
NSP was intended to teach the model sentence-level coherence and discourse relationships (helpful for tasks like QA and natural language inference). The model learns when two segments are likely contiguous vs. unrelated.

#### **Example NSP inputs & expected outputs**

1. `[CLS] The sky was dark and stormy . [SEP] Raindrops began to patter on the roof . [SEP]` → **IsNext = 1**  
2. `[CLS] She opened the ancient book carefully . [SEP] Later that day he fixed the kitchen sink . [SEP]` → **IsNext = 0**  
4. `[CLS] The committee approved the new policy . [SEP] The cat chased the laser pointer around the room . [SEP]` → **IsNext = 0**  
5. `[CLS] The patient recovered after the surgery . [SEP] The doctors were pleased with the test results . [SEP]` → **IsNext = 1**  
6. `[CLS] She loves gardening in spring . [SEP] The recipe calls for two cups of flour . [SEP]` → **IsNext = 0**

> **Implementation note:** NSP was part of original BERT pretraining; later work (e.g., RoBERTa) showed that removing NSP and adjusting other settings can improve performance. Still, NSP is how the original BERT learned sentence relationships.

---

### **MLM (Masked Language Modeling)**

MLM turns unlabelled text into a supervised prediction problem by *masking* some tokens and asking the model to predict the original tokens from their context. Key details from BERT's setup:

- **15%** of the tokens in each input are *selected* for prediction.  
- Of the selected tokens:  
  - **80%** are replaced with the special token `[MASK]`.  
  - **10%** are replaced with a random token from the vocabulary.  
  - **10%** are left unchanged (but still counted in the prediction loss).  
- The model must predict the original token at each selected position. This is trained with a standard cross-entropy objective across the vocabulary for each masked position.

#### **Why the three-way strategy?**

- Using `[MASK]` lets the model explicitly learn to predict hidden words from both left and right context (bidirectional).  
- Replacing some with **random tokens** and leaving some **unchanged** prevents the model from relying solely on the presence of `[MASK]` and encourages robustness to real-world inputs (because at inference time there will be no `[MASK]` tokens).

#### **Concrete MLM examples**

1. Input: `[CLS] The quick brown [MASK] jumped over the lazy dog . [SEP]` - (selected token replaced with `[MASK]`) → the model should predict `fox`.

2. Input: `[CLS] She poured the hot [MASK] into a mug . [SEP]` - (selected token replaced with `[MASK]`) → predict `tea`.

3. Input: `[CLS] He drove the old car to the nearest banana . [SEP]` - (selected token replaced with random token) → model must learn original should be `garage`.

4. Input: `[CLS] The patient reported a sharp pain in the chest . [SEP]` - (unchanged but selected) and model must output the same thing.

5. Input: `[CLS] The concert was canceled because of heavy [MASK] . [SEP]` → predict `rain`.

6. Input: `[CLS] A wireless mouse connects to a computer via [MASK] . [SEP]` → predict `bluetooth` or `USB` depending on context.

7. Input: `[CLS] The scientist analyzed the sample under an guitar microscope . [SEP]` → must recover `electron` or `optical` instead of `guitar`.

8. Input: `[CLS] Please save the changes before you close the [MASK] . [SEP]` → predict `file`.

9. Input: `[CLS] They sailed across the ocean to reach the island . [SEP]` → should do nothing.

10. Input: `[CLS] The new policy will come into effect next [MASK] . [SEP]` → predict `month` or `year`.

> MLM was typically applied on inputs that were *either single sentences or sentence pairs* (the same inputs used for NSP). So a single training example might be:
>
> `[CLS] Sentence A tokens (some masked) [SEP] Sentence B tokens (some masked) [SEP]`
>
> where both MLM and NSP losses are computed:

$$
Loss = Loss_{MLM} + Loss_{NSP}
$$

---

### **The Power of Pre-Training**

After going through this heavy pre-training journey, BERT emerges as a model that can do far more than just understand words in isolation. It learns whether two sentences belong to the same context, whether they share related meanings, and whether a word is being used in the correct form and position. It even knows how to replace a word with another that makes the sentence more meaningful.

In other words, BERT comes out of pre-training with a remarkable grasp of language: not just vocabulary, but also grammar, semantics, and context. This is why it can be so easily applied to a wide range of downstream NLP tasks such as:

* **Question Answering**
* **Sentiment Analysis**
* **Text Classification**
* **Named Entity Recognition (NER)**
* **Machine Translation**
* **Chatbot development**
* **Summarization**
* And much more.

The most impressive part? All of this was achieved using only a massive amount of *unlabelled* text data — books, Wikipedia, and beyond. What once seemed useless in machine learning suddenly became the foundation for one of the strongest language models ever built.

Of course, pre-training came with a high computational cost, but the payoff is that we now have access to these **pre-trained models**. Instead of training BERT from scratch, you can simply download it (e.g., via Hugging Face Transformers in Python), load all the already-trained parameters, and then just fine-tune it for your own task. Sometimes, that fine-tuning is as simple as adding a classifier on top or adjusting a few layers.

This is the brilliance of BERT: it starts with the full power of language understanding already in place, and you only need to guide it a little to specialize it for your application.

### **Implementation From Scratch**

This code builds and trains a **BERT-like model** specialized for the Masked Language Modeling (MLM) task using the `Transformer` class we had created in the previous chapter.  
It starts by creating a vocabulary from our handmade `mouse` dataset, adding special tokens such as `[MASK]`, `[PAD]`, and `[UNK]`.  
Then it generates masked training data by randomly hiding words in sentences and recording their positions,  
so the model can later learn to predict the missing tokens from context.  

The model itself is an **encoder-only Transformer**, mimicking BERT’s architecture. Input embeddings are passed through stacked encoder layers, and the output at masked positions is sent into an **MLM head**  
(a small prediction network) that guesses the original words.  

Training proceeds in mini-batches: the collate function pads sequences, aligns masks, and prepares the targets.  
During each epoch, the model receives masked sentences, predicts the hidden words, and updates its weights by minimizing cross-entropy loss.  

### **Testing Phase**

Finally, after training, the model is tested on sentences with `[MASK]` tokens.  
It predicts the most likely replacements, shows the filled-in sentences, and also reports the top-3 candidate words with probabilities.  
In short, this whole pipeline demonstrates how BERT can be **pre-trained on unlabeled text** by simply learning to fill in blanks. But don't forget that the training set we have here is super small compared with the one used in the original BERT model.

In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 80.9 MB/s eta 0:00:00


In [ ]:
# @title **Previous Chapter Code**

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
import gensim.downloader as api
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Load Word2Vec model
word2vec = api.load('glove-twitter-25')
print(f'Loaded {len(word2vec.key_to_index)} word embeddings with {word2vec.vector_size} dimensions')

# UTILITY FUNCTIONS
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def get_embedding(token):
    """Get word embedding with fallback handling"""
    token = token.lower().strip('.,!?;()[]')
    if token in word2vec.key_to_index:
        return word2vec[token]
    # Deterministic fallback for unknown words
    rnd = np.random.RandomState(abs(hash(token)) % (2**32))
    return rnd.normal(scale=0.1, size=(word2vec.vector_size,)).astype(np.float32)

def tokenize(sentence):
    """Simple tokenization"""
    return sentence.lower().split()

def embed_sentence(sentence):
    """Convert sentence to tensor of embeddings"""
    tokens = tokenize(sentence)
    embeddings = [get_embedding(token) for token in tokens]
    return torch.tensor(embeddings, dtype=torch.float32)

def pad_collate(batch):
    """Collate function for batching with padding"""
    sequences = [item[0] for item in batch]
    mouse_positions = [item[1] for item in batch]
    labels = [item[2] for item in batch]

    # Get dimensions
    lengths = [seq.size(0) for seq in sequences]
    max_len = max(lengths)
    embed_dim = sequences[0].size(1)

    # Create padded batch
    padded_seqs = torch.zeros(len(batch), max_len, embed_dim)
    attention_masks = torch.zeros(len(batch), max_len, dtype=torch.bool)

    for i, seq in enumerate(sequences):
        length = seq.size(0)
        padded_seqs[i, :length] = seq
        attention_masks[i, :length] = True

    return (padded_seqs, attention_masks,
            torch.tensor(mouse_positions), torch.tensor(labels))

class ScaledDotProductAttention(nn.Module):
    """
    Attention(Q, K, V) = softmax(QK^T / √d_k)V
    """
    def __init__(self, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

    def forward(self, Q, K, V, mask=None):
        """
        Args:
            Q: Query tensor (batch, heads, seq_len, head_dim)
            K: Key tensor   (batch, heads, seq_len, head_dim)
            V: Value tensor (batch, heads, seq_len, head_dim)
            mask: Already processed mask with -1e9 values (batch, heads, seq_len, seq_len)
        """
        # Get dimensions
        d_k = Q.size(-1)  # head_dim

        # Step 1: Compute QK^T / √d_k (scaled dot-product)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(d_k)

        # Step 2: Apply pre-processed mask if provided
        if mask is not None:
            scores = scores + mask  # mask already contains -∞ for invalid positions

        # Step 3: Apply softmax to get attention weights
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)

        # Step 4: Apply attention weights to values
        context = torch.matmul(attention_weights, V)

        return context, attention_weights

class MultiHeadAttention(nn.Module):
    """Multi-Head Attention using multiple Scaled Dot-Product Attention heads"""
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        assert embed_dim % num_heads == 0

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        # The core attention mechanism
        self.scaled_dot_product_attention = ScaledDotProductAttention(dropout)

        # Linear projections
        self.q_linear = nn.Linear(embed_dim, embed_dim, bias=False)
        self.k_linear = nn.Linear(embed_dim, embed_dim, bias=False)
        self.v_linear = nn.Linear(embed_dim, embed_dim, bias=False)
        self.out_linear = nn.Linear(embed_dim, embed_dim, bias=False)

        self._init_weights()

    def _init_weights(self):
        for module in [self.q_linear, self.k_linear, self.v_linear, self.out_linear]:
            nn.init.xavier_uniform_(module.weight)

    def forward(self, x, mask=None):
        batch_size, seq_len, embed_dim = x.size()

        # Linear projections and reshape for multi-head
        Q = self.q_linear(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_linear(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_linear(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # Process mask for attention scores
        processed_mask = None
        if mask is not None:
            # mask: (batch, seq_len) -> need (batch, heads, seq_len, seq_len)
            mask = mask.unsqueeze(1).unsqueeze(2)  # (batch, 1, 1, seq_len)
            processed_mask = torch.zeros_like(torch.matmul(Q, K.transpose(-2, -1)))
            processed_mask.masked_fill_(~mask, -np.inf)

        # Apply scaled dot-product attention
        context, attention_weights = self.scaled_dot_product_attention(Q, K, V, processed_mask)

        # Concatenate heads and apply final projection
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, embed_dim)
        output = self.out_linear(context)

        return output, attention_weights

class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding"""
    def __init__(self, embed_dim, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, embed_dim)
        position = torch.arange(0, max_len).unsqueeze(1).float()

        div_term = 1.0 / (10000 ** (torch.arange(0, embed_dim, 2).float() / embed_dim))

        pe[:, 0::2] = torch.sin(position * div_term)
        if embed_dim % 2 == 1:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])  # Remove last element for odd dimensions
        else:
            pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)

        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class FeedForward(nn.Module):
    """Position-wise Feed Forward Network"""
    def __init__(self, embed_dim, ff_dim, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(embed_dim, ff_dim)
        self.linear2 = nn.Linear(ff_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))

class EncoderBlock(nn.Module):
    """
    Single Transformer Encoder Block
    Architecture: Input -> Multi-Head Attention -> Add & Norm -> Feed Forward -> Add & Norm -> Output
    """
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        """
        Args:
            embed_dim: Dimension of token embeddings (d_model in paper)
            num_heads: Number of parallel attention heads
            ff_dim: Hidden dimension of feed-forward network (typically 4 * embed_dim)
            dropout: Dropout probability for regularization
        """
        super().__init__()
        self.attention = MultiHeadAttention(embed_dim, num_heads, dropout)
        self.feed_forward = FeedForward(embed_dim, ff_dim, dropout)
        self.norm1 = nn.LayerNorm(embed_dim)  # Layer norm after attention
        self.norm2 = nn.LayerNorm(embed_dim)  # Layer norm after feed-forward
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        """
        Forward pass implementing: MultiHeadAttn -> Add&Norm -> FFN -> Add&Norm

        Args:
            x: Input embeddings (batch_size, seq_len, embed_dim)
            mask: Optional attention mask (batch_size, seq_len)

        Returns:
            x: Transformed embeddings (batch_size, seq_len, embed_dim)
            attn_weights: Attention weights for visualization (batch_size, heads, seq_len, seq_len)
        """
        # Multi-head self-attention with residual connection and layer norm
        attn_output, attn_weights = self.attention(x, mask)
        x = self.norm1(x + self.dropout(attn_output))

        # Feed-forward with residual connection and layer norm
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))

        return x, attn_weights

class DecoderBlock(nn.Module):
    """
    Single Transformer Decoder Block
    Architecture: Input -> Masked Self-Attention -> Add&Norm -> Cross-Attention -> Add&Norm -> FFN -> Add&Norm
    """
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        """
        Args:
            similar to the encoder block
        """
        super().__init__()
        # Masked self-attention (decoder tokens attend to previous decoder tokens only)
        self.self_attention = MultiHeadAttention(embed_dim, num_heads, dropout)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.dropout1 = nn.Dropout(dropout)

        # Cross-attention (decoder attends to encoder outputs)
        self.cross_attention = MultiHeadAttention(embed_dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.dropout2 = nn.Dropout(dropout)

        # Position-wise feed-forward network
        self.feed_forward = FeedForward(embed_dim, ff_dim, dropout)
        self.norm3 = nn.LayerNorm(embed_dim)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, enc_out, src_mask=None, tgt_mask=None):
        """
        Forward pass implementing the three sub-layers of decoder block

        Args:
            x: Target embeddings (batch_size, tgt_len, embed_dim)
            enc_out: Encoder outputs (batch_size, src_len, embed_dim)
            src_mask: Source sequence mask (batch_size, src_len)
            tgt_mask: Target sequence mask - typically causal/triangular (batch_size, tgt_len)

        Returns:
            x: Transformed target embeddings (batch_size, tgt_len, embed_dim)
            attention_weights: Tuple of (self_attention_weights, cross_attention_weights)
        """
        # Masked self-attention: target tokens attend to previous target tokens only
        self_attn_output, self_attn_weights = self.self_attention(x, tgt_mask)
        x = self.norm1(x + self.dropout1(self_attn_output))

        # Cross-attention: target tokens (queries) attend to source tokens (keys/values)
        cross_attn_output, cross_attn_weights = self.cross_attention(x, src_mask, enc_out)
        x = self.norm2(x + self.dropout2(cross_attn_output))

        # Position-wise feed-forward network
        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout3(ff_output))

        return x, (self_attn_weights, cross_attn_weights)

class Transformer(nn.Module):
    """
    Complete Transformer Architecture
    Flexible implementation supporting:
    - Encoder-only models (like BERT): Set num_decoders=0
    - Decoder-only models (like GPT): Set num_encoders=0
    - Encoder-Decoder models (like original paper): Both > 0
    """
    def __init__(self, embed_dim, num_heads, num_encoders, num_decoders,
                 ff_dim, dropout=0.1, max_len=512):
        """
        Args:
            embed_dim: Model dimension (d_model in paper) - typically 512
            num_heads: Number of attention heads - typically 8
            num_encoders: Number of encoder blocks to stack (N in paper) - typically 6
            num_decoders: Number of decoder blocks to stack (N in paper) - typically 6
            ff_dim: Feed-forward hidden dimension - typically 4 * embed_dim = 2048
            dropout: Dropout probability for regularization
            max_len: Maximum sequence length for positional encoding
        """
        super().__init__()
        self.embed_dim = embed_dim
        self.pos_encoding = PositionalEncoding(embed_dim, max_len)

        # Encoder stack: N identical layers
        self.encoders = nn.ModuleList([
            EncoderBlock(embed_dim, num_heads, ff_dim, dropout)
            for _ in range(num_encoders)
        ])

        # Decoder stack: N identical layers (empty if encoder-only model)
        self.decoders = nn.ModuleList([
            DecoderBlock(embed_dim, num_heads, ff_dim, dropout)
            for _ in range(num_decoders)
        ])

    def forward(self, src, tgt=None, src_mask=None, tgt_mask=None):
        """
        Forward pass through the complete Transformer

        Args:
            src: Source embeddings (batch_size, src_len, embed_dim)
            tgt: Target embeddings (batch_size, tgt_len, embed_dim) - None for encoder-only
            src_mask: Source attention mask (batch_size, src_len)
            tgt_mask: Target attention mask (batch_size, tgt_len) - typically causal

        Returns:
            For encoder-only: (encoded_output, encoder_attention_weights)
            For encoder-decoder: (decoded_output, (encoder_attention_weights, decoder_attention_weights))
        """
        # Add positional encoding to source embeddings
        src = self.pos_encoding(src)
        all_enc_attn_weights = []

        # Pass through encoder stack
        for enc in self.encoders:
            src, attn_w = enc(src, src_mask)
            all_enc_attn_weights.append(attn_w)

        # If no decoders, return encoder output (BERT-style)
        if len(self.decoders) == 0:
            return src, all_enc_attn_weights

        # If decoders exist, process target sequence (original Transformer)
        tgt = self.pos_encoding(tgt)
        all_dec_attn_weights = []

        # Pass through decoder stack
        for dec in self.decoders:
            tgt, attn_w = dec(tgt, src, src_mask, tgt_mask)
            all_dec_attn_weights.append(attn_w)

        return tgt, (all_enc_attn_weights, all_dec_attn_weights)

# --- Create a comprehensive dataset with clear contextual differences ---

# Animal contexts
animal_templates = [
    "the {adj} mouse {action} in the {location}",
    "a mouse with {feature} {action} near the {animal_companion}",
    "the mouse {action} and {action2} in the {habitat}",
    "farmers found the mouse {action} in the {location}",
    "the {animal_companion} chased the mouse that {action}",
    "a {adj} mouse {action} with its {body_part}",
    "the mouse and the {animal_companion} {action} together",
    "wild mouse {action} when they {action2}"
]

# Device contexts
device_templates = [
    "the {adj} mouse {action} on the {surface}",
    "a mouse with {tech_feature} {action} near the {device}",
    "the mouse {action} when the {software} {software_action}",
    "users need the mouse to {action} the {interface_element}",
    "the {device} and mouse {action} via {connection}",
    "a {adj} mouse {action} with its {tech_component}",
    "the mouse and {device} {action} simultaneously",
    "wireless mouse {action} when they {tech_action}"
]

# Rich vocabulary for each context
animal_vocab = {
    'adj': ['small', 'tiny', 'hungry', 'scared', 'wild', 'brown', 'grey', 'quick'],
    'action': ['scurried', 'squeaked', 'nibbled', 'hid', 'ran', 'climbed', 'jumped', 'searched'],
    'action2': ['ate cheese', 'found crumbs', 'built nests', 'dug holes', 'gathered food'],
    'location': ['barn', 'attic', 'basement', 'kitchen', 'garden', 'field', 'warehouse'],
    'animal_companion': ['cat', 'hamster', 'rat', 'rabbit', 'dog', 'owl', 'snake'],
    'habitat': ['forest', 'meadow', 'farm', 'countryside', 'woods', 'grassland'],
    'feature': ['long tail', 'soft fur', 'small ears', 'tiny paws', 'whiskers'],
    'body_part': ['tail', 'paws', 'whiskers', 'teeth', 'ears', 'nose']
}

device_vocab = {
    'adj': ['wireless', 'optical', 'gaming', 'ergonomic', 'bluetooth', 'laser', 'mechanical'],
    'action': ['clicked', 'scrolled', 'connected', 'responded', 'tracked', 'moved', 'dragged'],
    'tech_action': ['lose signal', 'need batteries', 'update firmware', 'calibrate sensors'],
    'surface': ['desk', 'mousepad', 'table', 'workstation', 'surface', 'mat'],
    'device': ['keyboard', 'monitor', 'computer', 'laptop', 'pc', 'screen', 'webcam'],
    'tech_feature': ['scroll wheel', 'dpi settings', 'rgb lighting', 'programmable buttons'],
    'software': ['driver', 'software', 'system', 'application', 'program', 'interface'],
    'software_action': ['updated', 'crashed', 'loaded', 'responded', 'initialized'],
    'interface_element': ['icon', 'button', 'menu', 'window', 'cursor', 'link'],
    'connection': ['usb', 'bluetooth', 'wireless', 'cable', 'dongle', 'receiver'],
    'tech_component': ['sensor', 'battery', 'receiver', 'buttons', 'wheel', 'cable']
}

def generate_sentence(templates, vocab, label):
    template = np.random.choice(templates)
    # Fill template with random vocabulary
    sentence = template.format(**{key: np.random.choice(values)
                                for key, values in vocab.items()})
    return sentence, label

# Generate balanced dataset
dataset = []
n_samples_per_class = 800  # Total of 1600 sentences
set_seed()

for _ in range(n_samples_per_class):
    # Animal context
    sentence, label = generate_sentence(animal_templates, animal_vocab, 0)
    tokens = tokenize(sentence)
    if 'mouse' in tokens:
        mouse_pos = tokens.index('mouse')
        dataset.append((sentence, mouse_pos, label))

for _ in range(n_samples_per_class):
    # Device context
    sentence, label = generate_sentence(device_templates, device_vocab, 1)
    tokens = tokenize(sentence)
    if 'mouse' in tokens:
        mouse_pos = tokens.index('mouse')
        dataset.append((sentence, mouse_pos, label))

np.random.shuffle(dataset)

[==================================================] 100.0% 104.8/104.8MB downloaded
Loaded 1193514 word embeddings with 25 dimensions


In [ ]:
set_seed()

class BERTModel(nn.Module):
    """
    BERT-like model for Masked Language Modeling (MLM)
    Uses encoder-only Transformer architecture with MLM head
    """
    def __init__(self, vocab_size, embed_dim, num_heads, num_layers, ff_dim, dropout=0.1, max_len=512):
        super().__init__()
        self.embed_dim = embed_dim
        self.vocab_size = vocab_size

        # Token embedding layer (maps token indices to embeddings)
        self.token_embedding = nn.Linear(embed_dim, embed_dim)  # Since we use pre-trained embeddings

        # Encoder-only transformer (num_decoders=0)
        self.transformer = Transformer(
            embed_dim=embed_dim,
            num_heads=num_heads,
            num_encoders=num_layers,
            num_decoders=0,  # BERT is encoder-only
            ff_dim=ff_dim,
            dropout=dropout,
            max_len=max_len
        )

        # MLM head: predicts masked tokens
        self.mlm_head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(),
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, vocab_size)
        )
        self.mask_embedding = nn.Parameter(torch.randn(embed_dim) * 0.1)

    def forward(self, embeddings, mask=None, masked_positions=None):
        # Pass through transformer encoder
        encoded, attention_weights = self.transformer(embeddings, src_mask=mask)

        if masked_positions is not None:
            # Filter out invalid positions (-1)
            batch_size = masked_positions.size(0)
            valid_predictions = []

            for i in range(batch_size):
                valid_pos = masked_positions[i][masked_positions[i] >= 0]
                if len(valid_pos) > 0:
                    batch_masked_emb = encoded[i, valid_pos]  # (num_valid, embed_dim)
                    batch_pred = self.mlm_head(batch_masked_emb)
                    valid_predictions.append(batch_pred)

            if valid_predictions:
                predictions = torch.cat(valid_predictions, dim=0)
                return predictions, attention_weights

        return encoded, attention_weights

def create_vocabulary_from_dataset(dataset):
    """Create vocabulary from dataset sentences"""
    vocab = set()
    for sentence, _, _ in dataset:
        tokens = tokenize(sentence)
        vocab.update(tokens)

    # Add special tokens
    vocab.update(['[MASK]', '[PAD]', '[UNK]'])

    # Create mappings
    word_to_idx = {word: idx for idx, word in enumerate(sorted(vocab))}
    idx_to_word = {idx: word for word, idx in word_to_idx.items()}

    return word_to_idx, idx_to_word

def create_mlm_data(dataset, word_to_idx, mask_prob=0.15):
    """Convert dataset to MLM format with masked tokens"""
    mlm_data = []

    for sentence, mouse_pos, label in dataset:
        tokens = tokenize(sentence)

        # Create masked version
        masked_tokens = tokens.copy()
        masked_positions = []
        target_tokens = []

        # Randomly mask tokens (15% probability)
        for i, token in enumerate(tokens):
            if np.random.random() < mask_prob:
                masked_positions.append(i)
                target_tokens.append(word_to_idx.get(token, word_to_idx['[UNK]']))
                masked_tokens[i] = '[MASK]'

        if len(masked_positions) > 0:  # Only include if some tokens are masked
            mlm_data.append({
                'original_tokens': tokens,
                'masked_tokens': masked_tokens,
                'masked_positions': masked_positions,
                'target_tokens': target_tokens
            })

    return mlm_data

def embed_masked_sentence(masked_tokens, word_to_idx, model=None):
    """Convert masked tokens to embeddings"""
    embeddings = []
    for token in masked_tokens:
        if token == '[mask]':
            if model is not None:
                emb = model.mask_embedding.detach().cpu().numpy()
            else:
                emb = np.random.normal(0, 0.1, word2vec.vector_size).astype(np.float32)
        else:
            emb = get_embedding(token)
        embeddings.append(emb)

    return torch.tensor(embeddings, dtype=torch.float32)

def mlm_collate_fn(batch):
    """Collate function for MLM training"""
    max_len = max(len(item['masked_tokens']) for item in batch)
    max_masked = max(len(item['masked_positions']) for item in batch)
    embed_dim = word2vec.vector_size

    # Initialize tensors
    embeddings = torch.zeros(len(batch), max_len, embed_dim)
    attention_masks = torch.zeros(len(batch), max_len, dtype=torch.bool)
    masked_positions = torch.full((len(batch), max_masked), -1, dtype=torch.long)
    target_tokens = torch.full((len(batch), max_masked), -1, dtype=torch.long)

    for i, item in enumerate(batch):
        # Embeddings and mask
        emb = embed_masked_sentence(item['masked_tokens'], word_to_idx, bert_model)
        seq_len = emb.size(0)
        embeddings[i, :seq_len] = emb
        attention_masks[i, :seq_len] = True

        # Masked positions and targets
        num_masked = len(item['masked_positions'])
        masked_positions[i, :num_masked] = torch.tensor(item['masked_positions'])
        target_tokens[i, :num_masked] = torch.tensor(item['target_tokens'])

    return embeddings, attention_masks, masked_positions, target_tokens

# Create vocabulary and MLM dataset
print("Creating vocabulary and MLM dataset...")
word_to_idx, idx_to_word = create_vocabulary_from_dataset(dataset)
vocab_size = len(word_to_idx)
print(f"Vocabulary size: {vocab_size}")

mlm_dataset = create_mlm_data(dataset, word_to_idx, mask_prob=0.25)
print(f"MLM dataset size: {len(mlm_dataset)}")

# Split dataset
train_mlm, test_mlm = train_test_split(mlm_dataset, test_size=0.2, random_state=42)

# Initialize BERT model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
embed_dim = word2vec.vector_size

bert_model = BERTModel(
    vocab_size=vocab_size,
    embed_dim=embed_dim,
    num_heads=5,  # 25 % 5 = 0
    num_layers=3,
    ff_dim=embed_dim * 2,
    dropout=0.1
).to(device)

# Training setup
optimizer = torch.optim.AdamW(bert_model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss(ignore_index=-1)  # Ignore padded positions

# Training loop
batch_size = 32
num_epochs = 50
train_losses = []

print("Training BERT model...")
print("=" * 50)

for epoch in range(num_epochs):
    bert_model.train()
    epoch_losses = []

    # Shuffle training data
    np.random.shuffle(train_mlm)

    for i in range(0, len(train_mlm), batch_size):
        batch = train_mlm[i:i+batch_size]
        embeddings, masks, masked_pos, targets = mlm_collate_fn(batch)

        embeddings = embeddings.to(device)
        masks = masks.to(device)
        masked_pos = masked_pos.to(device)
        targets = targets.to(device)

        # Forward pass
        predictions, _ = bert_model(embeddings, masks, masked_pos)

        # Compute loss (only on valid masked positions)
        # Compute loss (flatten predictions and targets to match dimensions)
        predictions_flat = predictions.view(-1, vocab_size)
        targets_flat = targets[targets != -1]  # Remove padding

        if len(targets_flat) > 0:
            loss = criterion(predictions_flat[:len(targets_flat)], targets_flat)

            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_losses.append(loss.item())

    if len(epoch_losses) > 0:
        avg_loss = np.mean(epoch_losses)
        train_losses.append(avg_loss)

        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1:2d} | Loss: {avg_loss:.4f}")

Creating vocabulary and MLM dataset...
Vocabulary size: 142
MLM dataset size: 1420
Training BERT model...
Epoch  5 | Loss: 3.5530
Epoch 10 | Loss: 2.8691
Epoch 15 | Loss: 2.2898
Epoch 20 | Loss: 1.8321
Epoch 25 | Loss: 1.5239
Epoch 30 | Loss: 1.2905
Epoch 35 | Loss: 1.1501
Epoch 40 | Loss: 1.0375
Epoch 45 | Loss: 0.9493
Epoch 50 | Loss: 0.8989


In [ ]:
# Test sentences with blanks
test_sentences = [
    "the [MASK] mouse clicked on the screen",
    "a small mouse [MASK] in the barn",
    "the wireless mouse needs fully charged [MASK]",
    "the [MASK] chased the mouse through the kitchen",
    "my [MASK] mouse is fully charged",
    "a [MASK] mouse searched for cheese crumbs",
    "the [MASK] sensitivity settings need adjustment",
    "wild mouse builds nests in the [MASK]",
    "the bluetooth [MASK] pairs automatically with devices",
    "a gray mouse [MASK] a piece of cheese in the kitchen",
    "you need a [MASK] to navigate the interface",
    "a field mouse mother protects her [MASK] underground"
]

print("MLM PREDICTION RESULTS")
print("="*60)

bert_model.eval()
with torch.no_grad():
    for sentence in test_sentences:
        tokens = tokenize(sentence)
        mask_pos = tokens.index('[mask]')

        # Embed sentence
        embeddings = embed_masked_sentence(tokens, word_to_idx, bert_model).unsqueeze(0).to(device)
        mask = torch.ones(1, len(tokens), dtype=torch.bool).to(device)
        masked_positions = torch.tensor([[mask_pos]], dtype=torch.long).to(device)

        # Get predictions
        predictions, _ = bert_model(embeddings, mask, masked_positions)
        predicted_idx = predictions[0].argmax().item()  # Remove the second [0]
        predicted_word = idx_to_word[predicted_idx]
        confidence = F.softmax(predictions[0], dim=-1).max().item()  # Remove the second [0]

        # Get top 3 predictions
        k = min(3, predictions.size(-1))  # Handle case where vocab < 3
        top_3_indices = predictions[0].topk(k).indices  # Remove the second [0]
        top_3_words = [idx_to_word[idx.item()] for idx in top_3_indices]
        top_3_probs = F.softmax(predictions[0], dim=-1)[top_3_indices].tolist()  # Remove the second [0]

        filled_sentence = sentence.replace('[MASK]', predicted_word)
        sent = sentence.replace('[MASK]', '________')
        print(f"Original: {sent}")
        print(f"Filled:   {filled_sentence}")
        print(f"Top 3:    {[(word, f'{prob:.3f}') for word, prob in zip(top_3_words, top_3_probs)]}")
        print()

MLM PREDICTION RESULTS
Original: the ________ mouse clicked on the screen
Filled:   the wireless mouse clicked on the screen
Top 3:    [('wireless', '0.843'), ('wild', '0.026'), ('bluetooth', '0.012')]

Original: a small mouse ________ in the barn
Filled:   a small mouse jumped in the barn
Top 3:    [('jumped', '0.281'), ('hid', '0.148'), ('climbed', '0.122')]

Original: the wireless mouse needs fully charged ________
Filled:   the wireless mouse needs fully charged firmware
Top 3:    [('firmware', '0.338'), ('cursor', '0.105'), ('dongle', '0.071')]

Original: the ________ chased the mouse through the kitchen
Filled:   the rabbit chased the mouse through the kitchen
Top 3:    [('rabbit', '0.171'), ('owl', '0.118'), ('snake', '0.099')]

Original: my ________ mouse is fully charged
Filled:   my wild mouse is fully charged
Top 3:    [('wild', '0.288'), ('wireless', '0.236'), ('mouse', '0.223')]

Original: a ________ mouse searched for cheese crumbs
Filled:   a wild mouse searched for chee

### **Fine-Tuning Pretrained BERT**

When working with large language models, it quickly becomes clear that training a model **from scratch** for every new task is computationally prohibitive. Modern language models are large not by accident: we have already seen that, due to phenomena like **double descent**, increasing model capacity often leads to better generalization rather than overfitting. This naturally tempts us to scale models as much as possible. However, this benefit comes at the cost of **extreme computational and financial requirements**, making full training impractical in most real-world scenarios.

At this point, it is useful to recall a technique we discussed earlier in `Section05 - Chapter01`: Transfer Learning. In the computer vision setting, a deep network can be pretrained on a large and diverse image dataset. The early layers typically learn **general-purpose features** such as edges, textures, and simple shapes, which are largely task-independent. When adapting the model to a specific task, these early layers can be frozen, and only the later layers are trained. This approach, often referred to as **parameter fine-tuning**, allows us to achieve strong performance with relatively low computational cost.

Language models, however, behave differently.

Consider a pretrained model like **BERT**, which may consist of dozens or even hundreds of Transformer layers. A naive strategy would be to freeze most of the network and fine-tune only the top layers, similar to what is commonly done with CNNs. In practice, this approach is often ineffective for language understanding tasks.

The reason lies in the nature of linguistic information.

In vision models, early layers tend to capture **low-level, context-independent patterns**. Consider a simple image:  
an image of a **person sitting on a chair in a room**.  
Regardless of *why* the person is sitting (resting, waiting, thinking, or feeling sad) the image itself does not change. A CNN’s early layers will respond to edges, colors, shapes, and textures (the outline of the chair, the human shape, lighting contrasts). These features are largely **fixed and objective**, and their meaning does not shift based on interpretation or external context.

Now consider the **textual description** of the same scene:

> *“A person is sitting alone on a chair.”*

Unlike the image, this sentence can activate **very different meanings** depending on context.  
In a psychological context, it may suggest loneliness or isolation.  
In a medical context, it may imply fatigue or recovery.  
In a social context, it may indicate waiting or exclusion.  

Crucially, these interpretations are not superficial additions, they influence how *every word* in the sentence is understood. Even the earliest Transformer layers must already participate in modeling such abstract, contextual relationships. They are not merely detecting surface patterns, but are involved in constructing meaning that depends on background knowledge, intent, and context.

As a result, freezing most of the model and training only a small set of top layers often prevents the network from properly adapting its internal representations to a new task or domain. This seems to suggest that, unlike in vision, we might need to retrain the entire model for each new language task; an approach that is clearly infeasible.

This challenge motivates the development of **Parameter-Efficient Fine-Tunning (PEFT)** for large language models: methods that allow effective adaptation to new tasks while updating only a **small subset of parameters**, rather than retraining the entire network from scratch.

* **Adapters**:
They are small trainable neural modules inserted between the layers of a pretrained Transformer. During fine-tuning, the original model parameters are frozen, and only these adapter modules are updated. This allows the model to adapt to new tasks with minimal additional parameters and much lower computational cost, while preserving the knowledge stored in the pretrained weights.

* **Bias-Terms Fine-Tuning (BitFit):**
BitFit is a highly parameter-efficient fine-tuning method where **only the bias terms** of the model are updated, and all weight matrices remain frozen. Surprisingly, for many NLP tasks, adjusting just the biases is enough to achieve competitive performance. This works because biases can subtly shift activation patterns throughout the network without altering its core representations.

* **Reparametrization:**
Reparametrization refers to fine-tuning strategies where updates are not applied directly to the original weights, but instead to a **reparameterized form** of them. The idea is to express weight updates through a smaller set of parameters (often low-rank or constrained), reducing memory and computation while still allowing effective adaptation. LoRA is a well-known example built on this principle.

* **Prefix Tuning:**
Prefix tuning keeps the entire language model frozen and instead learns a small set of **trainable prefix vectors** that are prepended to the input sequence at each Transformer layer. These prefixes act as soft prompts that steer the model’s behavior toward a specific task, allowing task adaptation without modifying the model’s internal weights.

#### **Code Implementation**

This code adapts a pretrained **DistilBERT** model to classify sentences as **offensive** (1) or **polite** (0) using a small dataset of 40 labeled examples (20 offensive, 20 polite). Instead of fully fine-tuning the entire model, it applies **Bias-Terms Fine-Tuning (BitFit)**, a parameter-efficient strategy where only the bias parameters of the network are updated.

- **What It Does**:
  - **Loads DistilBERT**: Initializes a pretrained Transformer with a classification head.
  - **Freezes Most Parameters**: All weight matrices are frozen to preserve pretrained linguistic representations.
  - **Applies BitFit**: Only bias terms are made trainable, allowing task adaptation with minimal parameter updates.
  - **Preprocesses Text**: Tokenizes sentences into input IDs and attention masks.
  - **Trains Efficiently**: Fine-tunes bias parameters for 10 epochs using a learning-rate scheduler for stable optimization.
  - **Performs Inference**: Predicts labels by selecting the class with the highest logit.

- **Why This Works**:
  - Even small shifts in bias terms can meaningfully adjust activation patterns across the network.
  - BitFit drastically reduces computational cost and overfitting risk while retaining most of the pretrained model’s knowledge.

- **Key Components**:
  - Tokenization via `DistilBertTokenizer`.
  - Bias-only optimization using `AdamW`.

In [ ]:
!pip install transformers datasets tokenizers

In [ ]:
import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, logging, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from datasets import Dataset

# Suppress transformer warnings for cleaner output
logging.set_verbosity_error()
set_seed()

# Load tokenizer and model
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")  # Load DistilBERT tokenizer
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)  # Load DistilBERT for binary classification

# Freeze all the parameters
for param in model.parameters():
    param.requires_grad = False

# Unfreeze only the bias terms
for name, param in model.named_parameters():
    if "bias" in name:
        param.requires_grad = True

# Dataset: 20 offensive + 20 polite sentences (label 1=offensive, 0=polite)
sentiment_sentences = {
    "text": [
        # Offensive
        "You’re an idiot who can’t do anything right.",
        "This is the worst service I’ve ever seen.",
        "Get lost, you annoying jerk.",
        "Your ideas are stupid and useless.",
        "Nobody likes you, you’re a total loser.",
        "What a pathetic excuse for a human.",
        "You’re so dumb, it’s embarrassing.",
        "This place is a disgusting mess.",
        "Shut up, you’re making me sick.",
        "You’re a failure and always will be.",
        "I hate everything you say.",
        "You sound completely brainless.",
        "Your presence ruins the mood.",
        "Nobody respects a fool like you.",
        "You’re the worst person to work with.",
        "Every word you say is nonsense.",
        "You should be ashamed of yourself.",
        "I can’t stand your stupidity.",
        "You always mess things up.",
        "You make everyone miserable around you.",
        # Polite
        "Could you please help me with this task?",
        "I appreciate your effort in solving this.",
        "Thank you for your kind assistance.",
        "I’d be grateful if you could explain again.",
        "Your input is always very helpful.",
        "I value your thoughtful feedback.",
        "Please let me know how I can assist you.",
        "Your cooperation is greatly appreciated.",
        "I’m thankful for your patience and support.",
        "Could we discuss this in a friendly manner?",
        "It’s always a pleasure to collaborate with you.",
        "You handled that situation with great care.",
        "Your kindness never goes unnoticed.",
        "I admire your professionalism.",
        "I respect your thoughtful approach.",
        "I truly appreciate your dedication.",
        "Thank you for your generosity.",
        "I enjoy working with you on these tasks.",
        "Your assistance means a lot to me.",
        "I’m grateful for your constant support."
    ],
    "label": [1] * 20 + [0] * 20  # Binary labels
}

# Split data into train and validation sets (80/20)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    sentiment_sentences['text'], sentiment_sentences['label'], test_size=0.2, random_state=42
)

# Tokenize training and validation data
train_enc = tokenizer(train_texts, truncation=True, padding=True, return_tensors="pt")  # Tokenize train texts
val_enc = tokenizer(val_texts, truncation=True, padding=True, return_tensors="pt")  # Tokenize val texts

# Create TensorDatasets
train_dataset = torch.utils.data.TensorDataset(
    train_enc["input_ids"], train_enc["attention_mask"], torch.tensor(train_labels)
)  # Train dataset
val_dataset = torch.utils.data.TensorDataset(
    val_enc["input_ids"], val_enc["attention_mask"], torch.tensor(val_labels)
)  # Val dataset

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)  # Train loader
val_loader = DataLoader(val_dataset, batch_size=2)  # Val loader

# Initialize optimizer and scheduler
optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-2)  # AdamW with adjusted learning rate
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=10, num_training_steps=len(train_loader) * 10
)  # Linear scheduler

# Move model to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Select GPU or CPU
model.to(device)  # Move model to device

# Training loop
for epoch in range(10):
    model.train()  # Set model to training mode
    total_loss = 0
    for batch in train_loader:
        input_ids, attention_mask, labels = [x.to(device) for x in batch]  # Move batch to device
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)  # Forward pass
        loss = outputs.loss  # Compute loss
        total_loss += loss.item()  # Accumulate loss
        optimizer.zero_grad()  # Zero gradients
        loss.backward()  # Backpropagate
        optimizer.step()  # Update weights
        scheduler.step()  # Update learning rate
    avg_loss = total_loss / len(train_loader)  # Compute average loss

    # Validation
    model.eval()  # Set model to evaluation mode
    correct, total = 0, 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids, attention_mask, labels = [x.to(device) for x in batch]  # Move batch to device
            outputs = model(input_ids, attention_mask=attention_mask)  # Forward pass
            preds = torch.argmax(outputs.logits, dim=-1)  # Predict class
            correct += (preds == labels).sum().item()  # Count correct predictions
            total += labels.size(0)  # Count total samples
    val_acc = correct / total if total > 0 else 0  # Compute accuracy
    print(f"Epoch [{epoch+1:>2}/10], Train Loss: {avg_loss:.4f}, Val Accuracy: {val_acc*100:.2f}%")

Epoch [ 1/10], Train Loss: 0.6839, Val Accuracy: 50.00%
Epoch [ 2/10], Train Loss: 0.6509, Val Accuracy: 50.00%
Epoch [ 3/10], Train Loss: 0.5518, Val Accuracy: 87.50%
Epoch [ 4/10], Train Loss: 0.3676, Val Accuracy: 100.00%
Epoch [ 5/10], Train Loss: 0.3040, Val Accuracy: 87.50%
Epoch [ 6/10], Train Loss: 0.2770, Val Accuracy: 100.00%
Epoch [ 7/10], Train Loss: 0.1859, Val Accuracy: 100.00%
Epoch [ 8/10], Train Loss: 0.1881, Val Accuracy: 100.00%
Epoch [ 9/10], Train Loss: 0.1421, Val Accuracy: 100.00%
Epoch [10/10], Train Loss: 0.1308, Val Accuracy: 100.00%


In [ ]:
# Test dataset
new_sentences = [
    "I really appreciate the effort you put into this.",    # polite
    "You are such a useless idiot.",                        # offensive
    "Could you kindly help me with my homework?",           # polite
    "This is the dumbest thing I’ve ever heard.",           # offensive
    "Thank you so much for your assistance.",               # polite
    "I'm sure even your parents hate you.",                 # offensive
    "You should go to hell, if you aren't there yet.",      # offensive
    "I am so thankful of all you did for me."               # polite
]

# Tokenize test sentences
inputs = tokenizer(new_sentences, padding=True, truncation=True, return_tensors="pt")  # Tokenize test data
input_ids = inputs['input_ids'].to(device)  # Move input IDs to device
attention_mask = inputs['attention_mask'].to(device)  # Move attention mask to device

# Predict on test sentences
model.eval()  # Set model to evaluation mode
with torch.no_grad():
    outputs = model(input_ids, attention_mask=attention_mask)  # Forward pass
    preds = torch.argmax(outputs.logits, dim=-1).tolist()  # Predict classes (0 or 1)

# Print test results
print("Test Sentence Predictions:")
for sent, pred in zip(new_sentences, preds):
    label = "🚫Offensive" if pred == 1 else "✔️Polite"
    print(f"Sentence: {sent}")
    print(f"Result: {label}\n{'-'*25}")

Test Sentence Predictions:
Sentence: I really appreciate the effort you put into this.
Result: ✔️Polite
-------------------------
Sentence: You are such a useless idiot.
Result: 🚫Offensive
-------------------------
Sentence: Could you kindly help me with my homework?
Result: ✔️Polite
-------------------------
Sentence: This is the dumbest thing I’ve ever heard.
Result: 🚫Offensive
-------------------------
Sentence: Thank you so much for your assistance.
Result: ✔️Polite
-------------------------
Sentence: I'm sure even your parents hate you.
Result: 🚫Offensive
-------------------------
Sentence: You should go to hell, if you aren't there yet.
Result: 🚫Offensive
-------------------------
Sentence: I am so thankful of all you did for me.
Result: ✔️Polite
-------------------------


## **GPT (Jun 2018, OpenAI)**

The first **Generative Pretrained Transformer (GPT)** was introduced by OpenAI in June 2018 in their paper [Improving Language Understanding by Generative Pre-Training](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf) as a bold experiment: what if we simply took the *decoder* part of the Transformer, trained it as a generative model on a massive corpus of books, and then fine-tuned it for specific NLP tasks?  

The model itself was not huge by today’s standards, but it was a solid step forward:  
- **12 stacked Transformer decoder layers**.  
- **12 self-attention heads per layer**.  
- **Hidden size of 768 dimensions**, with a total of **117 million parameters**.  

GPT worked in a **unidirectional, left-to-right fashion**, predicting the next token in a sequence given all the tokens before it. This autoregressive training allowed it to learn coherent sentence structures and contextual word meanings.  

What truly made GPT remarkable was not just the architecture, but the proof of concept: after pretraining on the **BooksCorpus** dataset (800M words), the same model could be fine-tuned for a wide range of tasks. GPT achieved **state-of-the-art results on 9 out of 12 benchmark tasks** in NLP, including textual entailment, semantic similarity, and question answering — all by reusing the same pretrained backbone.  

For the first time, the community saw the real potential of large-scale pretraining: one model, trained once on raw text, could be adapted to many different downstream tasks. This simple but powerful idea set the stage for the rapid progress that followed in GPT-2, GPT-3, and beyond.  

### **Pre-Training Phase**

The heart of GPT lies in its **pre-training phase**, where the model is trained as a **unidirectional (causal) language model**. This means that, given a sequence of words, GPT only looks **from left to right** and learns to predict the *next word* at every step. For example, if the sequence is “The patient reported a sharp pain in the”, GPT is trained to guess the most likely next word, such as “chest.” Unlike BERT’s masked approach, GPT never sees “future” tokens during training, which makes it fundamentally a **generative model**.

What made this pre-training powerful was not just the left-to-right prediction, but the fact that it was done on a **massive dataset (BooksCorpus, with 7,000+ unpublished books)** and at scale with **117 million parameters**. This allowed GPT to learn general-purpose patterns of grammar, semantics, and reasoning simply by predicting the next word over and over. The model effectively captured how words, phrases, and even long-range dependencies in text align with each other. This broad linguistic knowledge meant GPT could be applied to many downstream tasks without needing to train a model from scratch.

Once the pre-training was complete, GPT entered the **fine-tuning phase**, where a smaller, labeled dataset for a specific task was used to adjust the model. By slightly updating parameters or adding task-specific classifiers, GPT could adapt its general knowledge to other tasks. In essence, pre-training gave GPT a **universal language backbone**, while fine-tuning allowed it to specialize.

### **Evolution of GPT Models**

* **GPT-1 (2018)**:
  The very first GPT had **117M parameters** and was trained on the **BooksCorpus** dataset. Its big contribution wasn’t raw power but the *idea*: show that a simple Transformer decoder, trained left-to-right on a large unlabeled corpus, could then be fine-tuned to achieve strong performance on multiple NLP benchmarks. GPT-1 was the proof of concept that large-scale pre-training truly worked.

* **GPT-2 (2019)**:
  GPT-2 scaled things up dramatically to **1.5B parameters** and was trained on **WebText**, a dataset of 8 million web pages. With this jump in scale, the model gained the ability to **generate long, coherent passages of text** that often stayed on topic for several paragraphs. GPT-2 also showed early signs of *zero-shot and few-shot learning*: it could perform tasks it was never explicitly trained on just by being given examples in its prompt. This was the first time the research community saw language models acting more like general-purpose tools rather than task-specific systems.

* **GPT-3 (2020)**:
  GPT-3 pushed the boundaries with a staggering **175B parameters** trained on a mix of web pages, books, Wikipedia, and more. This sheer scale led to **emergent abilities** that weren’t present in smaller models. For instance, GPT-3 could perform **arithmetic calculations, word scrambles, translation, and commonsense reasoning** — all without explicit task-specific training. It also made **few-shot prompting** a standard way to use language models, where the model learns a task simply from a handful of examples in the input text. GPT-3’s success marked the beginning of large language models becoming mainstream tools.

* **Beyond GPT-3 (InstructGPT & ChatGPT)**:
  Later refinements, such as **InstructGPT** (2022), introduced **reinforcement learning from human feedback (RLHF)** to align the model’s responses with human expectations. This step was crucial for making the model safer and more useful in real applications. **ChatGPT** (built on GPT-3.5 and later GPT-4) took this further by focusing on **dialogue**, fine-tuning the model to follow instructions, maintain context, and provide conversational answers. While the architecture still relied on the same decoder blocks, the training methodology gave it an entirely new “personality” suited for interactive use.

### **Code Implementation**

In [ ]:
from tokenizers import ByteLevelBPETokenizer
from datasets import load_dataset

# Step 1: Load dataset
dataset = load_dataset("roneneldan/TinyStories", split="train")
N = 100000
training_texts = [sample["text"] for sample in dataset.select(range(N))]

# Step 2: Save texts to file (tokenizers library needs files for training)
with open("train_texts.txt", "w", encoding="utf-8") as f:
    for line in training_texts:
        f.write(line + "\n")

# Step 3: Train ByteLevelBPETokenizer
tokenizer = ByteLevelBPETokenizer()
tokenizer.train(files="train_texts.txt", vocab_size=5000, min_frequency=2,
                special_tokens=["<PAD>", "<UNK>", "<BOS>", "<EOS>"])

# Save tokenizer
tokenizer.save_model(".", "tiny_bpe")

# Step 4: Reload tokenizer for use
from tokenizers.implementations import ByteLevelBPETokenizer as BPEImp
tokenizer = BPEImp("tiny_bpe-vocab.json", "tiny_bpe-merges.txt")

VOCAB_SIZE = tokenizer.get_vocab_size()
print("Vocab size:", VOCAB_SIZE)

# Fix the MultiHeadAttention class first
class MultiHeadAttention(nn.Module):
    """Multi-Head Attention using multiple Scaled Dot-Product Attention heads"""
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        assert embed_dim % num_heads == 0

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        # The core attention mechanism
        self.scaled_dot_product_attention = ScaledDotProductAttention(dropout)

        # Linear projections
        self.q_linear = nn.Linear(embed_dim, embed_dim, bias=False)
        self.k_linear = nn.Linear(embed_dim, embed_dim, bias=False)
        self.v_linear = nn.Linear(embed_dim, embed_dim, bias=False)
        self.out_linear = nn.Linear(embed_dim, embed_dim, bias=False)

        self._init_weights()

    def _init_weights(self):
        for module in [self.q_linear, self.k_linear, self.v_linear, self.out_linear]:
            nn.init.xavier_uniform_(module.weight)

    def forward(self, x, mask=None):
        batch_size, seq_len, embed_dim = x.size()

        # Linear projections and reshape for multi-head
        Q = self.q_linear(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_linear(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_linear(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # Process mask for attention scores - FIXED VERSION
        processed_mask = None
        if mask is not None:
            # Ensure mask has the right shape and type
            if mask.dim() == 2:  # (batch_size, seq_len)
                mask = mask.unsqueeze(1).unsqueeze(1)  # (batch_size, 1, 1, seq_len)
            elif mask.dim() == 3:  # (batch_size, seq_len, seq_len)
                mask = mask.unsqueeze(1)  # (batch_size, 1, seq_len, seq_len)

            # Expand mask to match number of heads
            mask = mask.expand(batch_size, self.num_heads, seq_len, seq_len)

            # Convert to the right format for attention
            processed_mask = torch.zeros_like(mask, dtype=torch.float32)
            processed_mask.masked_fill_(mask.bool(), -1e9)

        # Apply scaled dot-product attention
        context, attention_weights = self.scaled_dot_product_attention(Q, K, V, processed_mask)

        # Concatenate heads and apply final projection
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, embed_dim)
        output = self.out_linear(context)

        return output, attention_weights

class GPT(nn.Module):
    """GPT Model - Decoder-only Transformer"""
    def __init__(self, vocab_size, embed_dim, num_heads, num_layers, ff_dim, max_len=512, dropout=0.1):
        super().__init__()
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.max_len = max_len

        # Token embeddings
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.pos_encoding = PositionalEncoding(embed_dim, max_len)

        # Decoder stack
        self.decoders = nn.ModuleList([
            DecoderBlock(embed_dim, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])

        # Final projection to vocabulary
        self.output_projection = nn.Linear(embed_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

        # Initialize weights
        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.token_embedding.weight, mean=0.0, std=0.02)
        nn.init.normal_(self.output_projection.weight, mean=0.0, std=0.02)
        if self.output_projection.bias is not None:
            nn.init.zeros_(self.output_projection.bias)

    def create_causal_mask(self, seq_len):
        """Create causal mask for decoder (upper triangular mask)"""
        mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1)
        mask = mask.bool()
        return mask

    def forward(self, x, attention_mask=None):
        batch_size, seq_len = x.shape

        # Get token embeddings
        x = self.token_embedding(x)
        x = self.pos_encoding(x)
        x = self.dropout(x)

        # Create causal mask
        causal_mask = self.create_causal_mask(seq_len)
        causal_mask = causal_mask.to(x.device)

        # Process attention mask
        if attention_mask is not None:
            # Ensure mask is boolean
            attention_mask = attention_mask.bool()

            # Create padding mask from attention_mask
            padding_mask = ~attention_mask  # (batch_size, seq_len)
            padding_mask_2d = padding_mask.unsqueeze(1) | padding_mask.unsqueeze(2)  # (batch_size, seq_len, seq_len)

            # Combine with causal mask
            final_mask = causal_mask.unsqueeze(0) | padding_mask_2d  # (batch_size, seq_len, seq_len)
        else:
            final_mask = causal_mask.unsqueeze(0)  # (1, seq_len, seq_len)

        # Pass through decoder layers
        all_attention_weights = []
        for decoder in self.decoders:
            x = self._decoder_forward(decoder, x, final_mask)
            all_attention_weights.append(None)

        # Project to vocabulary
        logits = self.output_projection(x)

        return logits, all_attention_weights

    def _decoder_forward(self, decoder, x, mask):
        """Custom decoder forward for GPT (no encoder input)"""
        # Self-attention only (no cross-attention)
        self_attn_output, _ = decoder.self_attention(x, mask)
        x = decoder.norm1(x + decoder.dropout1(self_attn_output))

        # Skip cross-attention for GPT
        # Feed-forward network
        ff_output = decoder.feed_forward(x)
        x = decoder.norm3(x + decoder.dropout3(ff_output))

        return x

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, texts, tokenizer, max_length=128):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]

        # Encode with BPE
        tokens = self.tokenizer.encode(text).ids

        # Add BOS/EOS if you want
        tokens = [self.tokenizer.token_to_id("<BOS>")] + tokens + [self.tokenizer.token_to_id("<EOS>")]

        # Truncate & pad
        tokens = tokens[:self.max_length]
        padding_length = self.max_length - len(tokens)
        tokens = tokens + [self.tokenizer.token_to_id("<PAD>")] * padding_length

        input_seq = tokens[:-1]
        target_seq = tokens[1:]

        attention_mask = [1 if t != self.tokenizer.token_to_id("<PAD>") else 0 for t in input_seq]

        return (
            torch.tensor(input_seq),
            torch.tensor(attention_mask, dtype=torch.bool),
            torch.tensor(target_seq)
        )

def build_vocab(texts, min_freq=1):
    """Build vocabulary from texts"""
    token_counts = defaultdict(int)

    for text in texts:
        tokens = tokenize(text)
        for token in tokens:
            token_counts[token] += 1

    vocab = ['<PAD>', '<UNK>', '<BOS>', '<EOS>']

    for token, count in token_counts.items():
        if count >= min_freq:
            vocab.append(token)

    token_to_idx = {token: idx for idx, token in enumerate(vocab)}
    return vocab, token_to_idx

def generate_text(model, prompt, tokenizer, max_length=50, temperature=0.8):
    model.eval()

    # Encode prompt
    token_indices = tokenizer.encode(prompt).ids
    generated = token_indices.copy()

    with torch.no_grad():
        for _ in range(max_length):
            input_seq = torch.tensor([generated]).to(next(model.parameters()).device)
            attention_mask = torch.ones_like(input_seq, dtype=torch.bool)

            logits, _ = model(input_seq, attention_mask)
            next_token_logits = logits[0, -1, :] / temperature
            probs = F.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, 1).item()

            if next_token == tokenizer.token_to_id("<EOS>"):
                break

            generated.append(next_token)

    # Decode
    return tokenizer.decode(generated, skip_special_tokens=True)

Vocab size: 5000


In [ ]:
# Training configuration
set_seed(42)

# Model hyperparameters
EMBED_DIM = 64
NUM_HEADS = 16
NUM_LAYERS = 4
FF_DIM = 512
MAX_LENGTH = 128
BATCH_SIZE = 64
LEARNING_RATE = 2e-4
EPOCHS = 10

vocab, token_to_idx = build_vocab(training_texts, min_freq=1)
idx_to_token = {v: k for k, v in token_to_idx.items()}

# Create dataset and dataloader
dataset = TextDataset(training_texts, tokenizer, MAX_LENGTH)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GPT(vocab_size=VOCAB_SIZE,
                 embed_dim=EMBED_DIM,
                 num_heads=NUM_HEADS,
                 num_layers=NUM_LAYERS,
                 ff_dim=FF_DIM,
                 max_len=MAX_LENGTH).to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.token_to_id("<PAD>"))
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

# Training loop
print("Starting training...")
model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for batch_idx, (input_seq, attention_mask, target_seq) in enumerate(dataloader):
        input_seq, attention_mask, target_seq = (
            input_seq.to(device), attention_mask.to(device), target_seq.to(device)
        )

        optimizer.zero_grad()
        logits, _ = model(input_seq, attention_mask)

        loss = criterion(logits.view(-1, VOCAB_SIZE), target_seq.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    print(f"Epoch [{epoch+1}/{EPOCHS}], Train Loss: {avg_loss:.4f}")

Starting training...
Epoch [1/10], Train Loss: 5.2050
Epoch [2/10], Train Loss: 3.8091
Epoch [3/10], Train Loss: 3.4910
Epoch [4/10], Train Loss: 3.3109
Epoch [5/10], Train Loss: 3.1833
Epoch [6/10], Train Loss: 3.0837
Epoch [7/10], Train Loss: 3.0038
Epoch [8/10], Train Loss: 2.9377
Epoch [9/10], Train Loss: 2.8834
Epoch [10/10], Train Loss: 2.8381


In [ ]:
set_seed()
# Test generation
test_prompts = [
    "once upon a time there was a little",
    "one day I went to a forest and saw a",
    "when I saw her she was smiling and",
    "the treasure map led them to"
]

print("STORY GENERATION RESULTS")
print("="*50)

model.eval()
# Use tokenizer, not token_to_idx / idx_to_token
for prompt in test_prompts:
    generated = generate_text(model, prompt, tokenizer, max_length=50, temperature=0.7)
    print(f"\nPrompt: '{prompt}'")
    print(f"Generated: {generated.replace('\n', ' ')}")

STORY GENERATION RESULTS

Prompt: 'once upon a time there was a little'
Generated: once upon a time there was a little girl named Lucy. She was three years old and loved to play outside. She had a big ball that was very very pretty.  One day, Lucy found a big box in the room. It was shiny and round. She picked it it

Prompt: 'one day I went to a forest and saw a'
Generated: one day I went to a forest and saw a big, hairy bear. The owl was scared because it was very sad.  "I have a friend, I am the puppy. What is a pet?" asked the bear.  "I don't know," said the mouse. "

Prompt: 'when I saw her she was smiling and'
Generated: when I saw her she was smiling and she was very angry. She wanted to play in the park, so she asked her mom for help her.  Her mom said yes and she said no. The little girl was sad and didn't know what to do.  Her mom

Prompt: 'the treasure map led them to'
Generated: the treasure map led them to go to the beach. One day, the boat was road in the pond. The hurried

## **Closing This Chapter**

This chapter challenged one of the earliest intuitions in deep learning: that larger models inevitably overfit. By exploring the idea of **double descent**, we saw how increasing model capacity can actually *improve* generalization, giving rise to the era of **Large Models**.

Building on the Transformer architecture, we examined how language models exploit this capacity.  
Using the **encoder-only** design, we studied BERT and observed how contextual embeddings allow the same word to take on different meanings depending on usage. Training and fine-tuning demonstrated how these models internalize grammar, semantics, and intent, and how **transfer learning** turns general knowledge into task-specific intelligence with surprisingly little data.

Finally, by shifting to the **decoder-only** side, we explored generative modeling through GPT, where language understanding becomes language creation.

Together, these ideas reveal why modern NLP works, not because models are smaller or simpler, but because they are **large enough to learn language itself**.